[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadErlanggaKurniawan/ets-ml2-kelompok/blob/main/notebooks/04_track_C_telur_puyuh_slot.ipynb)

# 04 · Track C — Mutu Internal Kuning Telur Puyuh (*slot*, menunggu data)

Topik skripsi Rijal: EfficientNet-B0 sebagai *feature extractor* + XGBoost (hybrid) dengan label
**Yolk Index** (tinggi/diameter kuning telur). Setiap telur adalah **sampel destruktif**. Rencananya
2 suhu × 31 hari × 6 butir = 372 butir.

**Status:** citra belum tersedia di repo. Notebook ini sudah memuat seluruh alur:
1. ekstraksi *embedding* EfficientNet-B0 (1.280 dimensi),
2. model tunggal vs hybrid XGBoost, dengan dan tanpa SMOTE,
3. *active learning* bandit yang sama seperti notebook 03.

Selama data belum ada, `SMOKE_TEST = True` menjalankan alur dengan **data sintetis**. Angka dari mode itu
**bukan hasil penelitian** dan tidak boleh dikutip.

Format data yang dibutuhkan (taruh di Google Drive lalu ubah `EGG_DIR`):
```
EGG_DIR/labels.csv     image,yolk_index,storage_day,temperature
EGG_DIR/images/<image>
```
Ambang kelas YI harus ditetapkan dalam protokol Rijal **sebelum** pemodelan.

## 0. Instalasi & utilitas

In [ ]:
# Versi dikunci sama dengan uv.lock di repo supaya angka di Colab = angka di laporan
%pip install -q scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 optuna==5.0.0 \
    imbalanced-learn==0.14.2 rdkit==2026.9.1 shap==0.54.1

In [ ]:
import warnings, hashlib, json, time, urllib.request, zipfile, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import lightgbm as lgb
from xgboost import XGBClassifier, XGBRegressor
from sklearn.base import clone
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              StackingClassifier, StackingRegressor)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore", message="The `probability` parameter was deprecated")
SEED = 42
RAW, RESULTS = Path("data/raw"), Path("results")
RAW.mkdir(parents=True, exist_ok=True); RESULTS.mkdir(exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch(url, dest):
    import ssl, certifi                       # certifi: CA bundle yang sama di Colab dan macOS
    dest = Path(dest)
    if not dest.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "ets-ml2"})
        ctx = ssl.create_default_context(cafile=certifi.where())
        with urllib.request.urlopen(req, context=ctx) as r, open(dest, "wb") as f:
            f.write(r.read())
    return dest

def mean_sd(v):
    v = np.asarray(v, float)
    return f"{v.mean():.3f} ± {v.std(ddof=1) if len(v) > 1 else 0:.3f}"

In [ ]:
SMOKE_TEST = True                     # ubah ke False setelah data Rijal tersedia
EGG_DIR = Path("/content/drive/MyDrive/egg_yolk")
YI_THRESHOLDS = None                  # contoh: (lo, hi) dari protokol Rijal - JANGAN ditebak

## 1. Ekstraksi embedding EfficientNet-B0 (hanya jika data nyata tersedia)

In [ ]:
if not SMOKE_TEST:
    assert YI_THRESHOLDS is not None, "isi ambang Yolk Index dari protokol penelitian"
    import torch
    from PIL import Image
    from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
    w = EfficientNet_B0_Weights.IMAGENET1K_V1; net = efficientnet_b0(weights=w)
    net.classifier = torch.nn.Identity(); net.eval(); prep = w.transforms()
    labels = pd.read_csv(EGG_DIR / "labels.csv"); lo, hi = YI_THRESHOLDS
    labels["y"] = np.select([labels.yolk_index < lo, labels.yolk_index < hi], [0, 1], 2)
    with torch.no_grad():
        F = np.vstack([net(prep(Image.open(EGG_DIR / "images" / n).convert("RGB")).unsqueeze(0)).squeeze(0).numpy()
                       for n in labels["image"]])
    X, y = F, labels["y"].to_numpy()
else:
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=372, n_features=64, n_informative=12, n_classes=3,
                               weights=[0.2, 0.5, 0.3], random_state=SEED)
print(X.shape, np.bincount(y))

## 2. Model tunggal vs hybrid XGBoost, ± SMOTE (Stratified 5-fold)

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
models = {"LogReg (single)": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
          "XGBoost (hybrid)": XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8,
                                            colsample_bytree=0.5, n_jobs=4, random_state=SEED, verbosity=0)}
rows = []
for fold, (tr, te) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y)):
    for name, m in models.items():
        for smote in [False, True]:
            Xtr, ytr = SMOTE(random_state=SEED).fit_resample(X[tr], y[tr]) if smote else (X[tr], y[tr])
            rows.append({"model": name, "smote": smote,
                         "macro_f1": f1_score(y[te], clone(m).fit(Xtr, ytr).predict(X[te]), average="macro")})
pd.DataFrame(rows).groupby(["model", "smote"]).macro_f1.agg(["mean", "std"]).round(3)

## 3. *Active learning* bandit
Fungsi `run_active_learning` sama dengan notebook 03. Salin sel §1 dari notebook 03 ke sini setelah data
tersedia. Untuk telur, setiap label = satu butir yang dipecah, sehingga penghematan label berarti
penghematan sampel destruktif.